# Amazon Bedrock AgentCore Runtime e AgentCore Memory Agent com isolamento de Identidade

## Visão Geral

Este tutorial demonstra como criar seu primeiro agente habilitado com memória e isolamento de usuário usando o AgentCore Runtime e o AgentCore Memory. Você construirá um agente conversacional simples do tipo "Hello World" que lembra interações anteriores dentro de uma sessão, permitindo conversas mais naturais e contextuais com os usuários ao longo das interações.

A memória é um componente crítico para criar agentes conversacionais eficazes, pois permite que eles mantenham o contexto, lembrem preferências do usuário e forneçam respostas consistentes ao longo do tempo. Sem memória, seu agente teria que começar do zero a cada interação, levando a uma experiência de usuário fragmentada.

A implementação utiliza o Amazon Bedrock AgentCore Memory com propagação de identidade do usuário para particionar automaticamente a memória com base nas credenciais autenticadas do usuário, criando espaços de memória seguros e isolados para cada usuário individual.

### Detalhes do Tutorial


| Informação          | Detalhes                                                         |
|:--------------------|:-----------------------------------------------------------------|
| Tipo de tutorial    | Hello World / Introdução                                         |
| Tipo de agente      | Agente Conversacional Único                                      |
| Framework Agêntico  | Strands Agents                                                   |
| Modelo LLM          | Anthropic Claude Haiku 3.5                                      |
| Recursos principais | AgentCore Runtime, Integração com Memory                         |
| Complexidade        | Intermediária                                                    |
| SDK utilizado       | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### O Que Você Aprenderá

Neste tutorial, você aprenderá:
1. Como criar um recurso de memória para seu agente usando o AgentCore Memory
2. Como implementar hooks de memória para armazenar e recuperar o histórico de conversas
3. Como implantar seu agente no AgentCore Runtime para uso em produção escalável
4. Como testar seu agente com gerenciamento de sessões e verificar a persistência da memória
5. Como lidar com a identidade do usuário e garantir o isolamento de memória entre diferentes usuários


### Arquitetura

Este exemplo Hello World demonstra um agente conversacional simples implantado no AgentCore Runtime com integração de memória:

<div style="text-align:left">
    <img src="runtime-memory-identity.png" width="90%"/>
</div>


## 0. Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10 ou mais recente
* Credenciais AWS configuradas com permissões apropriadas para Bedrock, ECR, IAM e Cognito
* Acesso ao modelo Amazon Bedrock (Claude 3.5 Haiku)
* SDK do Amazon Bedrock AgentCore e dependências

Primeiro, vamos instalar as bibliotecas necessárias.

In [ ]:
!pip install -qUr requirements.txt

### Configurando o Ambiente

Vamos importar as bibliotecas necessárias e configurar nosso ambiente. Utilizaremos:
- `boto3` para interações com serviços AWS
- `bedrock_agentcore.memory` para gerenciar a memória do agente
- Diversas funções utilitárias para configurar a autenticação

In [ ]:
# Imports
import os
import jwt
import time
import boto3
import uuid
import logging
from bedrock_agentcore.memory import MemoryClient
from utils import setup_cognito_user_pool, reauthenticate_users

# Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2')
memory_client = MemoryClient(region_name=REGION)

## 1. Criando o Amazon Cognito User Pool

Nesta seção, criaremos um Amazon Cognito User Pool e usuários. O Cognito fornece autenticação de usuários e gerenciamento de identidade para nosso agente, garantindo que o histórico de conversas de cada usuário seja acessível apenas para aquele usuário.

A função `setup_cognito_user_pool` irá:
1. Criar um Cognito User Pool se ele não existir
2. Configurar app clients para autenticação
3. Criar 2 usuários de teste com senhas temporárias
4. Gerar tokens de acesso para teste

In [ ]:
print("Setting up Amazon Cognito user pool and users...")
cognito_config = setup_cognito_user_pool(region=REGION)
print("Cognito setup completed ✓")

## 2. Criando o Recurso de Memória

Nesta seção, criaremos um recurso de memória para nosso agente armazenar o histórico de conversas. A memória permite que o agente relembre interações passadas, mantenha o contexto e forneça respostas mais coerentes ao longo do tempo.

Para este exemplo, criaremos um recurso simples de memória de curto prazo sem estratégias adicionais de longo prazo. A memória armazenará todas as mensagens da conversa, ajudando nosso agente a lembrar interações anteriores ao continuar uma sessão após ela ter sido encerrada no AgentCore Runtime.

In [ ]:
from botocore.exceptions import ClientError

# Create unique identifier for this resource
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeIdentityMemoryAgent_{unique_id}"

try:
    # Create memory resource without strategies (short-term memory only)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # No strategies for short-term memory
        description="Short-term memory for AgentCore Runtime agent authenticated with AgentCore Identity",
        event_expiry_days=7, # Retention period for short-term memory
    )
    memory_id = memory['id']
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Show any errors during memory creation
    logger.error(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

## 3. Criando Seu Agente Habilitado com Memória

Nesta seção, construiremos nosso agente habilitado com memória usando o framework Strands Agents com hooks personalizados para integração com memória. Este agente manterá o contexto da conversa armazenando e recuperando mensagens do AgentCore Memory.

> **Por que a Memória é Importante**: As sessões no AgentCore Runtime expiram após um determinado tempo, o que exclui o contexto da conversa. Ao armazenar as conversas na memória, garantimos que as informações anteriores persistam entre as sessões, criando uma experiência fluida para os usuários mesmo após longos intervalos.

### Capacidades do Agente

Nosso agente irá:
1. Armazenar cada mensagem do usuário e do assistente na memória automaticamente
2. Recuperar o histórico de conversas anteriores ao continuar uma sessão existente
3. Manter o contexto ao longo de múltiplas interações com o mesmo usuário
4. Isolar conversas entre diferentes usuários através da verificação de identidade do usuário

### Componentes Principais da Nossa Implementação

#### 1. Memory Hook Provider
Nosso hook provider personalizado implementa:
- `on_agent_initialized`: Acionado quando o agente inicia, recupera o histórico de conversas do AgentCore Memory
- `on_message_added`: Acionado quando uma nova mensagem é adicionada à conversa, armazena-a no AgentCore Memory

#### 2. Inicialização do Agente
A função `initialize_agent`:
- Configura o hook de memória com a região correta
- Configura o agente com as variáveis de estado adequadas (memory_id, actor_id, session_id)
- Configura o system prompt para o LLM

#### 3. Verificação do Usuário
A função `get_user_sub`:
- Verifica um token de acesso do Cognito contra o JWKS e retorna o sub (ID único) do usuário.

#### 4. Handler do Ponto de Entrada
A função runtime_memory_agent:
- Analisa o payload de entrada e extrai a mensagem do usuário
- Verifica a identidade do usuário usando tokens JWT do Cognito
- Gerencia a inicialização do agente e o rastreamento de sessões
- Lida com a invocação do agente com o contexto adequado
- Retorna respostas formatadas para o ambiente de runtime

Vamos criar o arquivo do nosso agente:

In [ ]:
%%writefile runtime_identity_memory_agent.py
import os
import jwt
import json
import logging
from strands import Agent
from jwt import PyJWKClient
from typing import Dict, Any
from strands.models import BedrockModel
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

# Configure detailed logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# Initialize the agent core app
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
COGNITO_USER_POOL = os.getenv('COGNITO_USER_POOL')
REGION = os.getenv('AWS_REGION')

# Global agent instance - will be initialized with first request
agent = None

class MemoryHookProvider(HookProvider):
    """Custom hook provider to integrate with Bedrock Memory"""
    
    def __init__(self, region_name):
        logger.info(f"Initializing MemoryHookProvider with region {region_name}")
        self.memory_client = MemoryClient(region_name=region_name)
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts"""
        logger.info("Agent initialization hook triggered")
        
        memory_id = event.agent.state.get("memory_id")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"State values - memory_id: {memory_id}, actor_id: {actor_id}, session_id: {session_id}")
        
        missing_values = []
        if not memory_id:
            missing_values.append("memory_id")
        if not actor_id:
            missing_values.append("actor_id")
        if not session_id:
            missing_values.append("session_id")
            
        if missing_values:
            logger.warning(f"Missing required values: {', '.join(missing_values)}")
            return
        
        try:
            # First, check if the session exists by listing events with a limit of 1
            logger.info(f"Checking if session {session_id} exists...")
            session_exists = False
            try:
                events = self.memory_client.list_events(
                    memory_id=memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    max_results=1
                )
                session_exists = len(events) > 0
                logger.info(f"Session exists: {session_exists} (found {len(events)} events)")
            except Exception as e:
                logger.warning(f"Error checking session existence: {e}")
                # Assume no events exist and continue
                session_exists = False
            
            # If session doesn't exist, no need to load conversation history
            if not session_exists:
                logger.info(f"No existing conversation found for session {session_id}")
                return
            
            # Session exists, load the conversation history
            logger.info(f"Loading conversation history for existing session {session_id}")
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )
            
            if recent_turns:
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns from memory")
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message['role']
                        content = message['content']['text']
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info("✅ Added conversation context to system prompt")
            else:
                logger.info("No recent turns found for this session")
                
        except Exception as e:
            logger.error(f"❌ Memory load error: {e}", exc_info=True)
    
    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory"""
        logger.info("Message added hook triggered")
        
        memory_id = event.agent.state.get("memory_id")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"State values - memory_id: {memory_id}, actor_id: {actor_id}, session_id: {session_id}")
        
        missing_values = []
        if not memory_id:
            missing_values.append("memory_id")
        if not actor_id:
            missing_values.append("actor_id")
        if not session_id:
            missing_values.append("session_id")
            
        if missing_values:
            logger.warning(f"❌ Cannot save message - missing values: {', '.join(missing_values)}")
            return
            
        messages = event.agent.messages
        try:
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            message_role = last_message["role"]
            
            logger.info(f"Saving {message_role} message to memory: {message_content[:30]}...")
            
            self.memory_client.create_event(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                messages=[(message_content, message_role)]
            )
            logger.info("✅ Message saved to memory successfully")
        except Exception as e:
            logger.error(f"❌ Memory save error: {e}", exc_info=True)
    
    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering memory hooks")
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

def initialize_agent(actor_id, session_id):
    """Initialize the agent for first use"""
    global agent
    
    logger.info(f"Initializing agent for actor_id={actor_id}, session_id={session_id}")
    
    # Create model and memory hook
    logger.info(f"Creating model with ID: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)
    logger.info(f"Creating memory hook with region: {REGION}")
    memory_hook = MemoryHookProvider(region_name=REGION)
    
    # Create agent with proper initial state
    logger.info("Creating agent with memory hook")
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses.",
        state={
            "memory_id": MEMORY_ID,
            "actor_id": actor_id,
            "session_id": session_id
        }
    )
    logger.info(f"✅ Agent initialized with state: {agent.state.get()}")

def get_user_sub(access_token: str, region: str, user_pool_id: str) -> str:
    """
    Verifies a Cognito access token against JWKS and returns the user's sub (unique ID).

    :param access_token: The JWT access token string
    :param region: AWS region of the Cognito User Pool
    :param user_pool_id: The Cognito User Pool ID
    :return: The user's 'sub' claim if the token is valid
    :raises jwt.InvalidTokenError: If verification fails
    """
    access_token = access_token[7:]
    jwks_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/jwks.json"
    jwks_client = PyJWKClient(jwks_url)
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)

    decoded = jwt.decode(
        access_token,
        signing_key.key,
        algorithms=["RS256"],
        issuer=f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}",
        options={"require": ["exp", "iat", "iss", "token_use"]}
    )

    if decoded.get("token_use") != "access":
        raise jwt.InvalidTokenError("Token is not an access token")

    return decoded["sub"]

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    Main entry point for the memory-enabled agent
    
    Args:
        payload: The input payload containing user data
        context: The runtime context object containing session information
    """
    global agent
    
    # Log both payload and context info
    logger.info(f"Received payload: {payload}")
    logger.info(f"Context: {context}")
    logger.info(f"Context Auth: {context.request_headers.get('Authorization')}")
    logger.info(f"User Sub: {get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)}")
    logger.info(f"Context session_id: {context.session_id}")
    
    # Extract and validate required values
    user_input = payload.get("prompt")
    actor_id = get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)
    session_id = context.session_id  # Get session_id from context
    
    # Validate required fields
    if user_input is None:
        error_msg = "❌ ERROR: Missing 'prompt' field in payload"
        logger.error(error_msg)
        return error_msg
    
    # Initialize agent on first request
    if agent is None:
        logger.info("First request - initializing agent")
        initialize_agent(actor_id, session_id)
    else:
        logger.info("Using existing agent instance")
        # Update the session ID in case it changed
        if agent.state.get("session_id") != session_id:
            logger.info(f"Updating session ID to {session_id}")
            agent.state.set("session_id", session_id)
        if agent.state.get("actor_id") != actor_id:
            logger.info(f"Updating actor ID to {actor_id}")
            agent.state.set("actor_id", actor_id)
    
    # Invoke the agent with the user's input
    logger.info(f"Invoking agent with input: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"✅ Agent response: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("Starting AgentCore application")
    app.run()

## 4. Implantando no AgentCore Runtime

Nesta seção, implantaremos nosso agente no Amazon Bedrock AgentCore Runtime, um ambiente de runtime gerenciado para agentes que oferece escalabilidade e operações simplificadas. O AgentCore Runtime cuida da complexidade da infraestrutura, permitindo que você se concentre na lógica do seu agente em vez de preocupações com implantação.

Diferente dos métodos tradicionais de implantação que exigem configuração e gerenciamento manual de servidores, o AgentCore Runtime empacota automaticamente seu código em containers, os implanta na infraestrutura AWS e fornece endpoints HTTPS seguros para invocação. Essa abordagem garante que seu agente possa escalar conforme a demanda e operar de forma confiável em ambientes de produção.

### Por Trás dos Bastidores

Quando você implanta no AgentCore Runtime, várias coisas acontecem automaticamente:
1. Seu código é empacotado em uma imagem de container Docker
2. A imagem do container é enviada ao Amazon ECR (Elastic Container Registry)
3. Uma função AWS Lambda ou serviço de container é provisionado para executar seu agente
4. Endpoints do API Gateway são criados para acesso seguro
5. Roles e permissões IAM são configurados para operação segura

### O Que Você Precisa Saber

- **AgentCore Runtime** empacota seu agente em um container Docker e o implanta na infraestrutura gerenciada da AWS
- **Variáveis de Ambiente** configurarão nosso agente:
  - `MEMORY_ID`: O recurso de memória que criamos anteriormente
  - `MODEL_ID`: ID do modelo Claude 3.5 Haiku
  - `AWS_REGION`: Região AWS para implantação
  - `COGNITO_USER_POOL`: O Cognito user pool para autenticação

> 💡 **Dica**: O AgentCore starter toolkit cuida de todos os passos complexos de implantação para nós, incluindo roles IAM, repositórios ECR e builds de containers.

### Configurar a Implantação

Vamos configurar nossa implantação:

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_identity_memory_agent.py", 
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    non_interactive=True, 
    memory_mode="NO_MEMORY",
    request_header_configuration = {"requestHeaderAllowlist": ["Authorization"]},
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config.get("discovery_url"),
            "allowedClients": [cognito_config.get("client_id")]
        }
    }
)
response

### Iniciar o agente

Agora vamos iniciar nosso agente no AgentCore Runtime. Este passo pega nosso agente configurado e o implanta na infraestrutura gerenciada do AgentCore. Durante este processo, também estamos passando as variáveis de ambiente essenciais que nosso agente precisa: o ID da memória que criamos anteriormente, o ID do modelo a ser usado, a região AWS e o ID do Cognito user pool para autenticação.

Uma vez implantado, nosso agente estará acessível através de um endpoint seguro que podemos invocar com mensagens de usuários. O endpoint será protegido pela autenticação do Cognito, garantindo que apenas usuários autorizados possam acessar nosso agente.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": memory_id,
        "MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
        "AWS_REGION": REGION,
        "COGNITO_USER_POOL": cognito_config["pool_id"]
    }
)

### Verificar status da implantação

Vamos verificar o status da implantação do nosso agente. Isso pode levar alguns minutos enquanto o AgentCore Runtime constrói seu container, provisiona os recursos necessários e implanta seu agente na infraestrutura AWS. Faremos polling do status a cada 10 segundos até que a implantação seja concluída.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Current status: {status}")

if status == 'READY':
    print("✅ Agent successfully deployed!")
else:
    print(f"❌ Deployment ended with status: {status}")

## 5. Testando Seu Agente

Agora que nosso agente está implantado, vamos testá-lo enviando mensagens e verificando se ele consegue lembrar interações anteriores. Também testaremos se diferentes usuários possuem contextos de memória isolados, garantindo que a conversa de um usuário não seja visível para outro usuário.

**Notas Importantes sobre Gerenciamento de Sessões**

- **Gerenciamento de Sessões**: Embora o AgentCore Runtime gere automaticamente um ID de sessão caso nenhum seja fornecido, é recomendável gerenciar explicitamente os IDs de sessão em sua aplicação. Isso lhe dá melhor controle sobre:
  - Continuar conversas após timeouts de sessão
  - Criar novas sessões quando apropriado (ex.: usuário inicia uma nova conversa)
  - Lidar com múltiplas conversas paralelas com o mesmo usuário
  - Implementar políticas de expiração de sessão baseadas nas necessidades da sua aplicação

- **Persistência de Memória**: Mesmo que uma sessão expire no AgentCore Runtime, nosso agente pode recuperar conversas anteriores do AgentCore Memory quando uma nova sessão começa com o mesmo usuário.

Vamos primeiro definir uma função auxiliar para validar tokens JWT:

In [ ]:
def test_user_memory_isolation():
    """
    Test that each user has isolated memory in AgentCore.
    
    This test verifies that:
    1. Each user's conversation is stored separately
    2. The agent remembers previous interactions with each user
    3. User data is not shared between different users
    """
    print("\n" + "=" * 50)
    print("USER MEMORY ISOLATION TEST")
    print("=" * 50)
    
    # Create session IDs for testuser1 and testuser2
    testuser1_session_id = f"agent-session-testuser1-{int(time.time())}"
    testuser2_session_id = f"agent-session-testuser2-{int(time.time())}"
    
    testuser1_token = cognito_config["bearer_tokens"]["testuser1"]
    testuser2_token = cognito_config["bearer_tokens"]["testuser2"]
    
    # Step 1: testuser1 shares her favorite color
    print("\n" + "-" * 50)
    print("STEP 1: First user shares personal information")
    print("-" * 50)
    print("testuser1: \"My favorite color is purple.\"")
    
    response1 = agentcore_runtime.invoke(
        {"prompt": "My favorite color is purple."},
        session_id=testuser1_session_id,
        bearer_token=testuser1_token
    )
    print(f"Agent: \"{response1['response']}\"")
    
    # Step 2: testuser2 shares his favorite food
    print("\n" + "-" * 50)
    print("STEP 2: Second user shares different information")
    print("-" * 50)
    print("testuser2: \"My favorite food is pizza.\"")
    
    response2 = agentcore_runtime.invoke(
        {"prompt": "My favorite food is pizza."},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token
    )
    print(f"Agent: \"{response2['response']}\"")
    
    # Step 3: testuser1 asks about her color
    print("\n" + "-" * 50)
    print("STEP 3: First user tests agent's memory")
    print("-" * 50)
    print("testuser1: \"What did I say my favorite color was?\"")
    
    response3 = agentcore_runtime.invoke(
        {"prompt": "What did I say my favorite color was?"},
        session_id=testuser1_session_id,
        bearer_token=testuser1_token
    )
    print(f"Agent: \"{response3['response']}\"")
    
    # Step 4: testuser2 asks about his food
    print("\n" + "-" * 50)
    print("STEP 4: Second user tests agent's memory")
    print("-" * 50)
    print("testuser2: \"What's my favorite food?\"")
    
    response4 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite food?"},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token
    )
    print(f"Agent: \"{response4['response']}\"")
    
    # Step 5: testuser1 asks about food (shouldn't know)
    print("\n" + "-" * 50)
    print("STEP 5: Testing memory isolation (first user)")
    print("-" * 50)
    print("testuser1: \"What's my favorite food?\"")
    
    response5 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite food?"}, 
        session_id=testuser1_session_id,
        bearer_token=testuser1_token
    )
    print(f"Agent: \"{response5['response']}\"")
    
    # Step 6: testuser2 asks about color (shouldn't know)
    print("\n" + "-" * 50)
    print("STEP 6: Testing memory isolation (second user)")
    print("-" * 50)
    print("testuser2: \"What's my favorite color?\"")
    
    response6 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite color?"},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token
    )
    print(f"Agent: \"{response6['response']}\"")
    

In [ ]:
test_user_memory_isolation()

## Conceitos Principais

Neste tutorial, você aprendeu vários conceitos importantes para construir agentes habilitados com memória usando o AgentCore:

1. **Integração com Memória**: Como usar o Amazon Bedrock Memory para armazenar o histórico de conversas entre sessões, permitindo que seu agente mantenha o contexto ao longo do tempo mesmo quando as sessões expiram.

2. **Gerenciamento de Sessões**: Como usar IDs de sessão para organizar conversas e recuperar o histórico relevante quando um usuário retorna, criando uma experiência fluida.

3. **Implantação no AgentCore**: Como implantar seu agente em um ambiente de runtime de produção que lida com escalabilidade, segurança e gerenciamento de infraestrutura automaticamente.

4. **Hooks de Memória**: Como implementar hooks personalizados que se integram com serviços de memória, permitindo armazenar e recuperar o histórico de conversas em pontos específicos do ciclo de vida do agente.

5. **Identidade e Privacidade do Usuário**: Como usar autenticação para garantir que o histórico de conversas de cada usuário seja privado e isolado dos demais usuários.

Esses conceitos fornecem uma base para construir agentes mais complexos com memória persistente e capacidades sofisticadas de gerenciamento de conversas.

## Limpeza (Opcional)

Se você não precisa mais dos recursos criados neste tutorial, pode limpá-los para evitar cobranças desnecessárias na AWS. Isso inclui:

1. O agente do AgentCore Runtime
2. O repositório ECR contendo a imagem do container do agente
3. O recurso de memória que armazena o histórico de conversas

Vamos primeiro identificar nossos recursos:

In [ ]:
# Get resource identifiers
if 'launch_result' in locals():
    print(f"Agent ID: {launch_result.agent_id}")
    print(f"ECR Repository: {launch_result.ecr_uri.split('/')[1]}")
else:
    print("Launch results not available")

In [ ]:
# Only run this cell if you want to delete all resources

# 1. Delete the AgentCore Runtime
if 'launch_result' in locals() and hasattr(launch_result, 'agent_id'):
    try:
        agentcore_control_client = boto3.client(
            'bedrock-agentcore-control',
            region_name=REGION
        )
        
        runtime_delete_response = agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result.agent_id,
        )
        print(f"✅ Deleted AgentCore Runtime: {launch_result.agent_id}")
    except Exception as e:
        print(f"❌ Error deleting AgentCore Runtime: {e}")
else:
    print("No AgentCore Runtime to delete")

# 2. Delete the ECR repository
if 'launch_result' in locals() and hasattr(launch_result, 'ecr_uri'):
    try:
        ecr_client = boto3.client(
            'ecr',
            region_name=REGION
        )
        
        repository_name = launch_result.ecr_uri.split('/')[1]
        response = ecr_client.delete_repository(
            repositoryName=repository_name,
            force=True  # Force deletion even if it contains images
        )
        print(f"✅ Deleted ECR repository: {repository_name}")
    except Exception as e:
        print(f"❌ Error deleting ECR repository: {e}")
else:
    print("No ECR repository to delete")

# 3. Delete the memory resource
if 'memory_id' in locals() and memory_id:
    try:
        memory_client = MemoryClient(region_name=REGION)
        memory_client.delete_memory_and_wait(memory_id=memory_id)
        print(f"✅ Deleted memory resource: {memory_id}")
    except Exception as e:
        print(f"❌ Error deleting memory resource: {e}")
else:
    print("No memory resource to delete")

# 4. Delete the Cognito User Pool and associated resources
if 'cognito_config' in locals() and cognito_config and 'pool_id' in cognito_config:
    try:
        cognito_client = boto3.client('cognito-idp', region_name=REGION)
        
        # Get the user pool ID
        pool_id = cognito_config['pool_id']
        
        # List and delete all user pool clients
        clients_response = cognito_client.list_user_pool_clients(
            UserPoolId=pool_id,
            MaxResults=60
        )
        
        for client in clients_response.get('UserPoolClients', []):
            client_id = client['ClientId']
            cognito_client.delete_user_pool_client(
                UserPoolId=pool_id,
                ClientId=client_id
            )
            print(f"✅ Deleted User Pool Client: {client_id}")
        
        # Delete the user pool itself
        cognito_client.delete_user_pool(
            UserPoolId=pool_id
        )
        print(f"✅ Deleted Cognito User Pool: {pool_id}")
        
    except Exception as e:
        print(f"❌ Error deleting Cognito resources: {e}")
else:
    print("No Cognito resources to delete")

print("\n✅ Cleanup complete")

## Parabéns!

Você construiu e implantou com sucesso seu primeiro agente habilitado com memória usando o Amazon Bedrock AgentCore Runtime, AgentCore Identity e AgentCore Memory! Este agente demonstra várias capacidades importantes:

1. **Persistência de Memória**: Seu agente consegue lembrar conversas anteriores.
2. **Identidade do Usuário**: Seu agente mantém históricos de conversa separados para diferentes usuários
3. **Infraestrutura Gerenciada**: Seu agente roda em infraestrutura gerenciada pela AWS, escalando automaticamente conforme necessário

### Próximos Passos

Agora que você entende o básico, pode aprimorar seu agente de várias formas:

1. **Adicionar Ferramentas**: Aprimore seu agente com ferramentas como calculadoras, conectores de banco de dados ou chamadas de API para permitir que ele execute ações além da conversa
2. **Melhorar a Memória**: Implemente estratégias de memória mais sofisticadas com memória de longo prazo
3. **Construir uma Interface**: Crie uma interface web ou mobile para seu agente usando frameworks como React, Flutter ou Swift
4. **Adicionar Lógica de Negócios**: Integre seu agente com sistemas de negócios como CRMs, bases de conhecimento ou ferramentas internas